In [1]:
from IPython.display import HTML

HTML('''<script>
code_show=true; 
function code_toggle() {
 if (code_show){
 $('div.input').hide();
 } else {
 $('div.input').show();
 }
 code_show = !code_show
} 
$( document ).ready(code_toggle);
</script>
<form action="javascript:code_toggle()"><input type="submit" value="Click here to toggle on/off the raw code."></form>''')

# ETL Walkthrough for SUD Bag Rejects Analysis

## Outline
 
> This notebook provides a comprehensive overview of ETL process behind the SUD Bag Rejects Analysis dashboard. The notebook only covers the part of the process related to generation of the two main fact tables behind the dashboard:
 * SUD_BAGGER_TOTALS
 * SUD_BAGGER_REJECTS
 
>*Authors:*
 * Name, email

>*In case of issues/questions reach out to:*
 * Nik Namestnikov (namestnikov.nk@pg.com)
 * Vaishali Pawar (pawar.vp.4@pg.com )
 * Sibomana, Philippe (sibomana.p@pg.com)


## Setting up the environment

### Imports

In [1]:

import pandas as pd
import pyodbc as pyodbc

import queries
import logging

import sys
import os
import logging
import sys
import argparse
import pytz
from os.path import join, split 

from datetime import datetime, timedelta
from tomlkit import parse, dumps, loads
from typing import List, Tuple

import historian

from sud_tools import *
from sud_utils import *
from tool_utils import *


### Reading in the configuration file


The configuration file contains all parameters required for the orchestration of the ETL process. This data is stored in toml file, path to which is provided as a parameter of an ETL job. In case if a configuration file is not provided the *ValueError* is raised.

In [2]:
cfg_file_nm = 'etl_config.toml'
cfg_file_path = join(os.getcwd(), cfg_file_nm)
# checking if config file exists
if not os.path.isfile(cfg_file_path):
    raise ValueError(f'No config file was found at: {cfg_file_path}')
else:
    # reading in the config file
    cfg = loads(open(cfg_file_path).read())

# Main ETL

### Configuration

In this section all configuration parameters required for the orchestration of the ETL are getting initialized with the values provided in the configuration file. The ETL process is carried out for each SUD site seperatly, thus following 2 sections outline General and Site specific configuration settings for the ETL.
1. General Configuration
2. Site Specific Configuration

#### 1. General configuration:


General configuration includes parameters related to job (e.g. identifier, name), logs (path to logs), datalab db connection parameters and Power BI refresh parameters as shown in below sections.


##### 1.1 Job-related parameters
Every job results in one or a few tables being populated with the new portion of data. The summary of the extract is logged in the ```job_summary_table_nm``` and further used for the incremental data loading.

In [3]:
site_cd = 'tak'
job_id = 10*cfg['job']['id']
site_job_id_offset = cfg['sites'][site_cd]['job_id_offset']
job_id = job_id + site_job_id_offset

job_nm = ': '.join((cfg['job']['name'], site_cd))
job_summary_table_nm = cfg['job']['job_summary_table_nm']

In [4]:
# logs_path = cfg['job']['path_to_logs']

# # cheking if the logs folder exist
# if not os.path.exists(logs_path):
#     print(f'INFO: Didn not find logs folder at {logs_path}. Creating it...')
#     os.makedirs(os.path.join(logs_path))

# # creating log file to write to 

# log_create_dttm = datetime.now().strftime('%Y-%m-%d %H-%M-%S')
# log_file_nm = f"{cfg['job']['name']}_{job_id}_{log_create_dttm}.log"
# log_file_path = os.path.join(logs_path, log_file_nm)

# log_file = open(log_file_path, 'w+')
# log_file.close()

# print(f'Redirecting logging and stdout to {log_file_path}')

# # setting up logging to a file
# logging.basicConfig(format='[%(asctime)s] : %(levelname)s : %(funcName)s : %(lineno)d - %(message)s',
#                     level=logging.INFO,
#                     filename=log_file_path)
# sys.stdout = open(log_file_path, 'a')

##### 1.2 Setting up the connection to DatalabDB

In [5]:
# datalab integration
datalab_cfg = cfg['datalab_db'].copy()

# setting up run configuration with parameters from cfg file
datalab_db_conn = get_mssql_conn_string(**datalab_cfg['connection'])
datalab_db_access_url = r"{}".format(datalab_cfg['access_token_url']['access_token_url'])
datalab_db_access_token = get_azure_sql_db_access_token(datalab_db_access_url)

pack_rejects_table_nm = datalab_cfg['output_tables']['pack_rejects_table_nm']
pack_production_table_nm = datalab_cfg['output_tables']['pack_production_table_nm']

pack_rejects_agg_freq = datalab_cfg['output_tables']['pack_rejects_agg_freq']
pack_production_agg_freq = datalab_cfg['output_tables']['pack_prod_agg_freq']

print(f'Will be populating {pack_rejects_table_nm}, {pack_production_table_nm} tables')
print(f'Datalab connection parameters: {datalab_db_conn}')

Will be populating SUD_BAGGER_REJECTS, SUD_BAGGER_TOTALS tables
Datalab connection parameters: Server=azpg-sqlserver-fhcenganalyticsdatalab.database.windows.net; Database=DatalabDB; Driver=ODBC Driver 17 for SQL Server;


##### 1.3 Parameter for power BI refresh

Setting up the parameters to trigger the refresh of the dataset on Power BI side to load the new data to the backend of the dashboard.

In [6]:
pbi_refresh_url = cfg['powerbi']['api_url']
pbi_refresh_app = cfg['powerbi']['app_name']

##### 1.4 Fetching the tag list information

Full list of tags for each site is fetched from the excel table. Path to the file is provided in the configuration file.
Note, that only enabled tags are extracted/used.

In [7]:
# reading the tag list file
tags_list_path = cfg['job']['path_to_tag_list']
print(f'Path to tag list table: {tags_list_path}')

# cheking if the logs folder exist
if not os.path.exists(tags_list_path):
    print(f'INFO: Didn not find logs folder at {tags_list_path}.')
    raise ValueError('Was not able to file tags list. Aborting...')
else:
    tags_list_df = pd.read_excel(tags_list_path)
    tags_list_df = tags_list_df.loc[tags_list_df['Enabled'] == 1].copy()
    print(f'Shape of the tags list table: {tags_list_df.shape}')

Path to tag list table: F:\Ecosystem Non-OneDrive\Development Area\MVP046 SUD Bag Tool\Data\Input\tags_list.xlsx
Shape of the tags list table: (142, 7)


In [8]:
tags_list_df

,Site,Module,Reject,Reject code,PLC tag,Historian tag,Enabled
0,Lima,Machine,Missing Gusset,10,RT_Reject[1],LXXX_Mespack_Missing_Gusset,1
1,Lima,Machine,Film Splice,20,RT_Reject[2],LXXX_Mespack_Film_Splice,1
2,Lima,Machine,Zipper Splice,30,RT_Reject[3],LXXX_Mespack_Zipper_Splice,1
3,Lima,Machine,Slider Present,40,RT_Reject[4],LXXX_Mespack_Slider_Present,1
4,Lima,Machine,Machine Index,50,RT_Reject[5],LXXX_Mespack_Machine_Index,1
...,...,...,...,...,...,...,...
143,Takasaki,Machine,Good Total,0,RT_Reject[38],LXXX_Mespack_Good_Bag_Total,1
144,Takasaki,Machine,Reject Total,0,RT_Reject[39],LXXX_Mespack_Reject_Bag_Total,1
145,Takasaki,Machine,Changeover Reject Total,0,RT_Reject[40],LXXX_Mespack_Changeover_Reject_Total,1
146,Takasaki,Machine,2D & Markem,,MarkemRejectCounter.ACC.,LXXX_Mespack_2D_Markem_Reject,1


#### 2. Site Specific Configuration

Site specific configuration will include the parameters for getting site and line specific information from configuration file. This is shown in below subsections.




In [9]:
site_cfg = cfg['sites'][site_cd].copy()

site_name = site_cfg['site_name']
site_enabled = site_cfg['enabled']
site_agile_enabled = site_cfg['agile_enabled']
site_project_enabled = site_cfg['project_enabled']
site_job_id_offset = site_cfg['job_id_offset']
site_history_buffer_days = site_cfg['history_buffer_days']

site_servers = site_cfg['servers'].copy()
site_lines = site_cfg['lines']['lines']
site_lines_dim = site_cfg['lines']['lines_dim']
site_lines_agile = site_cfg['lines']['lines_agile']
line_maping_dict = dict(zip(site_lines, site_lines_dim))

site_historian_tags = site_cfg['tags'].copy()

site_tz = site_servers['timezone']
site_dttm_format = site_servers['dttm_format']
site_historian_source = site_servers['historian']['source']
site_history_depth = site_servers['historian']['history_depth_days']
site_days_to_retake = site_history_depth + site_history_buffer_days

site_tags = tags_list_df.loc[tags_list_df['Site'] == site_name, 'Historian tag'].unique()
site_tags = ['_'.join(t.split('_')[1:]) for t in site_tags]


In [10]:
print(f'Site name: {site_name}')
print(f'Site enabled: {site_enabled}')
print(f'Site agile enabled: {site_agile_enabled}')
print(f'Site project enabled: {site_project_enabled}')

print(f'Site lines: {site_lines}')
print(f'Site lines dimension: {site_lines_dim}')
print(f'Site lines agile: {site_lines_agile}')


Site name: Takasaki
Site enabled: True
Site agile enabled: False
Site project enabled: False
Site lines: ['L113', 'L114', 'L215', 'L314', 'L316']
Site lines dimension: ['L113', 'L114', 'SUTK-025', 'SUTK-027', 'SUTK-034']
Site lines agile: []


### Reading in the SUD Site and Line dimention tables

In [11]:
query_to_execute = "SELECT * FROM SUD_SITES"
sites_dim_df = pd.read_sql(sql=query_to_execute, con=get_db_connection(datalab_db_conn, datalab_db_access_token))
print(f'SUD sites dimention table loaded, shape: {sites_dim_df.shape}')

query_to_execute = "SELECT * FROM SUD_LINES"
lines_dim_df = pd.read_sql(sql=query_to_execute, con=get_db_connection(datalab_db_conn, datalab_db_access_token))
print(f'SUD lines dimention table loaded, shape: {lines_dim_df.shape}')


SUD sites dimention table loaded, shape: (5, 10)
SUD lines dimention table loaded, shape: (73, 13)


In [12]:
job_step_start_dttm = datetime.now()

## ETL Process 


**[Adjust]**
The data is extracted from Historian. The site historian sources are proficy and wonderware. The historian are *SQL Server databases* and the extraction is performed consecutively for each site. **[There is wonderware historian and Proficy - needs to be updated]**

We are generating 2 main tables SUB_BAGGER_TOTALS and SUD_BAGGER_REJECTS. In both cases, the process consists of the 3 main steps: Extract, Transform, and Load. Following are details for each step:
   
### Step 1: Extract
The Extract process consists of 2 steps:
   1. Defining timespan of the new extract.
   1. Extracting data.
   
### Step 1.1: Defining the timespan for the new extract

**[Adjust]**
1. This process consists of following steps:
    1. The most recent date we have in the historical data for corresponding site is defined.
    2. This date is further used to determine the time frame for the new extract from the source system.
    3. For each of the tables, the extraction time frame is defined by start and end timestamps
    4. Start time is defined as:
        1. Last available date selected from SUD_ETL_RUN_SUMMARY table if its not None. 
        2. current date, if its None.
    5. The end date defined as: one day forward of the current timestamp (based on the date of the ETL execution)



### Step 1.2 Extracting the data from source system
**[Adjust]** 
* To extract data from source system, we run the query on the database side to extract 'raw' data and further transform it



### Step 2: Transforming the data

The data is transformed to the format suitable for the analysis and visualisation in the dashboard. Namely:
1. Reject counters are cleaned and fixed to calculate number of reject bags at any given moment
2. Information on machine speed, agile and project run as well as other line state attributes are mapped to every bag reject
3. Site and line dimention identifiers are added
4. Data is aggregated 

### Step 3: Loading new extracts to the DatalabDB

**[Adjust]**
Too detailed. Simplify a bit and drop out the details.

* There are a two main options to perform the loading of the historical data
    1. *Truncate & Load*: you delete most recent data from the historical table and insert the data from the extract
    2. *Upsert or Update & Insert*: you update the data with the up-to-date values for the records shared between the historical data and new extract. Upsert query upserts dataframe to the target table in DB based on provided primary key. With *Insert*, you insert the new data which is only available in the new extract.
* We are using the Upsert approach.
* To do *Upsert or Update & Insert* query, we first create temporary table in the database where we put the data from the new extract and then perform Insert & Update operation.  This operation is performed if and only if we've extracted any data from on premis systems.

Following steps describes the incremental loading process:
1. We are extracting data from the source,transform it and putting into the target tables(Ex. SUD_BAGGER_REJECTS, SUD_BAGGER_TOTALS, etc.)
1. We are going to target table, taking the most recent data available. This gives us last timestamp available in this table. We will use this last date available as a start date for extracting new data from the source, and we will do a extract and load it into the tables.
2. For each incremental loading if its successful, we are generating run summary records and inserting it back to SUD_ETL_RUN_SUMMARY table. The format of this table is as below:


SUD_ETL_RUN_SUMMARY: columns and its meaning
1. job_id : job identifier
2. job_nm : job name
3. table_nm : Reference to the table that is populated by this job, this could be one or more tables ex. SUD_BAGGER_REJECTS, SUD_BAGGER_TOTALS. 
4. start_time : start time of executing the job
5. end_time : end time of job execution
6. rows_processed : Number of records that have been added to target table duing one incremental loading.
7. status : status of job execution, this could be success/fail. 
8. error_message : In case if the status is fail, we will get the error message.
9. colid : This is column id which is used during incremental loading.
10. coldttm : column date time
11. last_dttm_available : This is the timestamp of the most recent data available in the target table. We will use this timestamp as a start date for extracting new data from the source. 
12. last_id_available : This is id of the most recent data available in the target table. 


## ETL process execution



### Initial setup

The site_enabled == True indicates that the site is enabled. If its enabled, we will check the proficy historian source and extract the data from the source. 

There are following 3 main steps of tags:

1. Tags used to index or map every rejects to a particular state of the line (e.g. machine speed)
2. Tags used to extract the data for rejects happening on the line for every reject reason
3. Tags used to extract the data about the summarized/total number of rejected and produced bags



In [13]:
if site_enabled == True:
        if site_historian_source == 'proficy':
            # set up

            non_agile_tags = [t for t in site_tags if t not in site_historian_tags['tags_agile'] + site_historian_tags['tags_projects']]
            agile_tags = [t for t in site_tags if t in site_historian_tags['tags_agile']]
            project_tags = [t for t in site_tags if t in site_historian_tags['tags_projects']]

            site_line_tags = get_tags_list(lines=site_lines, sensors=non_agile_tags, sep='_', 
                                           topic=site_servers['historian']['proficy']['topic'])
            if agile_tags:
                site_agile_tags = get_tags_list(lines=site_lines_agile, sensors=agile_tags, sep='_', 
                                                topic=site_servers['historian']['proficy']['topic'])
            else: 
                site_agile_tags = []
            
            if project_tags:
                site_project_tags = get_tags_list(lines=site_lines, sensors=project_tags, sep='_', 
                                                 topic=site_servers['historian']['proficy']['topic'])
            else: 
                site_project_tags = []

            index_tags = [t for t in site_line_tags if '_'.join(t.split('_')[1:]) in site_historian_tags['tags_index']]
            tags_total_prod = [t for t in site_line_tags if t.endswith('Good_Bag_Total')]
            tags_total_rejects = [t for t in site_line_tags if not t.endswith('Good_Bag_Total') and 'total' in t.lower()]
            tags_rejects = [t for t in site_line_tags if (t not in index_tags + tags_total_prod + tags_total_rejects) 
                                                        or 'changeover' in t.lower()] 

            historian.use_context('REST', 
                    client_id='historian_public_rest_api', 
                    client_password='publicapisecret',
                    app_id='sudanalytics.im', 
                    app_password='phoenix2021SUD', # change to environmental variable
                    port=site_servers['historian']['proficy']['port'],
                    verify_certificate=True)


### Extract, transform and load for generating SUD_BAGGER_REJECTS table.

#### Extract

1. Defining timespan for the new extract



Below code describes the process of defining timespan for the new extract. 

Here the extract window is defined which will gets the dataframe which contains the record with the last results of etl run for the SUD_BAGGER_REJECTS table. The timespan is defined based on the information logged in the `job_summary_table_nm`. The last available date is selected from this dataframe. If it is not None, this is selected as a start date for extracting a new data. This date would be further used to determine the time frame for the new extract from the source system. If last available date is None, Start date is specified as the current date.



In [16]:
if site_enabled == True:
        if site_historian_source == 'proficy':
            # define extract window
            last_run_df = get_last_run_df(table_nm=pack_rejects_table_nm,
                        db_conn=get_db_connection(datalab_db_conn, datalab_db_access_token),
                        job_nm=job_nm)

            if not last_run_df.empty:
                last_available_dttm = last_run_df['last_dttm_available'][0]
                last_available_id = last_run_df['last_id_available'][0]
            else:
                last_available_dttm = None
                last_available_id = None

            start_dttm = datetime.now() if last_available_dttm is None else last_available_dttm
            start_dttm = start_dttm - timedelta(days=site_days_to_retake)
            start_time = start_dttm
            end_time = datetime.now()
            print(f'Extracting data between: {start_time} and {end_time}')

        

Extracting data between: 2023-03-06 19:00:00 and 2023-03-17 10:15:32.012529


In [15]:
#start_time = start_time - timedelta(days=13, hours=12, minutes=0)
start_time = pd.to_datetime('2023-02-20 06:00:00')
end_time = datetime.now()
print(start_time, end_time)

2023-02-20 06:00:00 2023-03-17 10:15:28.307248


In [76]:
# start_time = pd.to_datetime('2023-02-02 06:00:00')
# end_time = pd.to_datetime('2023-02-16 06:00:00')


In [77]:
###

2. Extracting data

We will extract machine state tags, rejects, and line states from the data source. We will use get_tag_values function to extract the tag values, rejects and machine status tags.

In [78]:
if site_enabled == True:
        if site_historian_source == 'proficy':
            #extraction
            ## extracting machine status tags
                machine_status_extract_df = historian.get_tag_values(
                    site_servers['historian']['proficy']['server_name'],
                    start_time=start_time - timedelta(days=1),
                    end_time=end_time,
                    filter_name=index_tags)
                print(f'Shape of machine speed extract: {machine_status_extract_df.shape}')
                
                machine_status_df = put_to_wonderware_format(machine_status_extract_df, inplace=False)
                machine_status_df = machine_status_df.assign(DateTime = machine_status_df['DateTime'].dt.tz_convert(site_tz).dt.tz_localize(None))
                machine_status_df = add_site_line_tag(machine_status_df, site=site_name)
                machine_status_df = machine_status_df.drop(columns=['tag'])

                ## extract rejects
                mespack_rejects_extract_df = historian.get_tag_values(
                    site_servers['historian']['proficy']['server_name'],
                    start_time=start_time,
                    end_time=end_time,
                    filter_name=tags_rejects)
                print(f'Shape of bag rejects extract: {mespack_rejects_extract_df.shape}')

                ## extract line state
                extract_dttm = datetime.strftime(start_time.date() - timedelta(days=1), '%Y%m%d %H:%M:%S') 
                query_to_execute = queries.EXTRACT_LINE_STATE.format(extract_dttm, tuple(site_lines_dim))

                line_state_map_df = pd.read_sql(sql=query_to_execute, con=get_db_connection(datalab_db_conn, datalab_db_access_token))
                print(f'Line state was extracted, shape: {line_state_map_df.shape}')

                index_cols = ['line_state_id', 'line', 'site']
                line_state_map_df = (line_state_map_df
                                    .set_index(index_cols)
                                    .stack()
                                    .to_frame('DateTime')
                                    .reset_index()
                                    .drop(columns=[f'level_{len(index_cols)}'])
                                    )
        
            
                
            

Shape of machine speed extract: (1280, 2)
TRANSFORM: Shape after addition of columns: (1280, 6)
Shape of bag rejects extract: (441880, 2)
Line state was extracted, shape: (18906, 5)


In [79]:
tags_rejects

['TAKF-PACKING.L113_Mespack_Missing_Gusset',
 'TAKF-PACKING.L113_Mespack_Film_Splice',
 'TAKF-PACKING.L113_Mespack_Zipper_Splice',
 'TAKF-PACKING.L113_Mespack_Slider_Present',
 'TAKF-PACKING.L113_Mespack_Machine_Index',
 'TAKF-PACKING.L113_Mespack_US_Bottom_Suction',
 'TAKF-PACKING.L113_Mespack_US_Top_Suction',
 'TAKF-PACKING.L113_Mespack_US_Bag_Open',
 'TAKF-PACKING.L113_Mespack_US_Scale_Discharge',
 'TAKF-PACKING.L113_Mespack_US_Filling_Disabled',
 'TAKF-PACKING.L113_Mespack_DS_Bottom_Suction',
 'TAKF-PACKING.L113_Mespack_DS_Top_Suction',
 'TAKF-PACKING.L113_Mespack_DS_Bag_Open',
 'TAKF-PACKING.L113_Mespack_DS_Scale_Discharge',
 'TAKF-PACKING.L113_Mespack_DS_Filling_Disabled',
 'TAKF-PACKING.L113_Mespack_Camera_DS_Eyemark',
 'TAKF-PACKING.L113_Mespack_Camera_DS_Nic',
 'TAKF-PACKING.L113_Mespack_Camera_DS_Left_Crush_Width',
 'TAKF-PACKING.L113_Mespack_Camera_DS_Left_Crush Height',
 'TAKF-PACKING.L113_Mespack_Camera_DS_Left_Crush_Pos',
 'TAKF-PACKING.L113_Mespack_Camera_DS_Right_Crush_

In [80]:
print('Extarcted historian data:')
mespack_rejects_extract_df.head()

Extarcted historian data:


Value  \
Tag                                      Timestamp                                
TAKF-PACKING.L113_Mespack_Missing_Gusset 2023-03-16 10:58:25.765000+00:00     0   
                                         2023-03-16 10:35:48.775000+00:00     0   
                                         2023-03-16 09:50:18.001000+00:00     0   
                                         2023-03-16 06:03:34.681000+00:00     4   
                                         2023-03-16 06:03:31.680000+00:00     2   

                                                                          Quality  
Tag                                      Timestamp                                 
TAKF-PACKING.L113_Mespack_Missing_Gusset 2023-03-16 10:58:25.765000+00:00       3  
                                         2023-03-16 10:35:48.775000+00:00       3  
                                         2023-03-16 09:50:18.001000+00:00       3  
                                         2023-03-16 06:03:34.681000+00:00       3  
                                         2023-03-16 06:03:31.680000+00:00       3

In [81]:
mespack_rejects_df = transform_counters_extract(mespack_rejects_extract_df, site_name=site_name, local_tz=site_tz)
                ## add machine speed
mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df, 
                            dim_df=machine_status_df.rename(columns={'Value':'Machine_Speed'}), 
                            dim_id='Machine_Speed', fact_dttm='DateTime', dim_dttm='DateTime', 
                            group_by=['site', 'line'])
mespack_rejects_df

TRANSFORM: Shape of original paking rejects data: (441713, 3)
TRANSFORM: Step 1 - Removing cases where the counter values were not logged (NaNs) or negative
TRANSFORM: # removed records 0
TRANSFORM: Shape after step: (441713, 3)
TRANSFORM: Step 2 - Removing cases, where counter has dropped down in between two resets (historian outage issue)
TRANSFORM: # removed records 16
TRANSFORM: Shape after step: (441697, 4)
TRANSFORM: Step 3 - Removing cases where the counter was not changing
TRANSFORM: # removed records 4367
TRANSFORM: Shape after step: (437330, 5)
TRANSFORM: Step 4 - Removing cases where the counter reports zero
TRANSFORM: # removed records 14896
TRANSFORM: Shape after step: (422434, 5)
TRANSFORM: Shape after addition of columns: (422434, 6)


,DateTime,Value,rejects_qty,line,tag,site,Machine_Speed
1831,2023-02-20 16:18:38.259,646.0,646.0,L113,Changeover_Reject_Total,Takasaki,40
6919,2023-02-20 16:18:38.259,6.0,6.0,L113,DS_Bottom_Suction,Takasaki,40
8511,2023-02-20 16:18:38.259,304.0,304.0,L113,DS_Filling_Disabled,Takasaki,40
17214,2023-02-20 16:18:38.259,3.0,3.0,L113,DS_Scale_Discharge,Takasaki,40
21159,2023-02-20 16:18:38.259,5.0,5.0,L113,US_Bottom_Suction,Takasaki,40
...,...,...,...,...,...,...,...
422429,2023-02-21 11:56:18.004,149.0,149.0,L314,DS_Filling_Disabled,Takasaki,35
422430,2023-02-21 11:56:18.004,149.0,149.0,L314,US_Filling_Disabled,Takasaki,35
422431,2023-02-21 10:32:19.780,244.0,244.0,L316,Changeover_Reject_Total,Takasaki,29
422432,2023-02-21 10:32:19.780,122.0,122.0,L316,DS_Filling_Disabled,Takasaki,29


In [82]:
## add line state
mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df.assign(line = mespack_rejects_df['line'].map(line_maping_dict)), 
                            dim_df=line_state_map_df, 
                            dim_id='line_state_id', fact_dttm='DateTime', dim_dttm='DateTime', 
                            group_by=['site', 'line'])
mespack_rejects_df

,DateTime,Value,rejects_qty,line,tag,site,Machine_Speed,line_state_id
0,2023-02-20 16:18:38.259,646.0,646.0,L113,Changeover_Reject_Total,Takasaki,40,23632/0
1,2023-02-20 16:18:38.259,6.0,6.0,L113,DS_Bottom_Suction,Takasaki,40,23632/0
2,2023-02-20 16:18:38.259,304.0,304.0,L113,DS_Filling_Disabled,Takasaki,40,23632/0
3,2023-02-20 16:18:38.259,3.0,3.0,L113,DS_Scale_Discharge,Takasaki,40,23632/0
4,2023-02-20 16:18:38.259,5.0,5.0,L113,US_Bottom_Suction,Takasaki,40,23632/0
...,...,...,...,...,...,...,...,...
422429,2023-02-21 11:56:18.004,149.0,149.0,SUTK-027,DS_Filling_Disabled,Takasaki,35,23670/0
422430,2023-02-21 11:56:18.004,149.0,149.0,SUTK-027,US_Filling_Disabled,Takasaki,35,23670/0
422431,2023-02-21 10:32:19.780,244.0,244.0,SUTK-034,Changeover_Reject_Total,Takasaki,29,23671/0
422432,2023-02-21 10:32:19.780,122.0,122.0,SUTK-034,DS_Filling_Disabled,Takasaki,29,23671/0


#### Transform

In the transform step, the data is transformed to the format suitable for the analysis and visualisation in the dashboard. Following steps are taken in the process of transforming the data:

1. Transforming counters
2. Adding reference to:
      1. Agile
      2. Project
      3. Line state
3. Adding information on machine speed.
4. Adding site and line dimension identifiers.
5. Aggregating data.

In [83]:
if site_enabled == True:
        if site_historian_source == 'proficy':
            # transform
                mespack_rejects_df = transform_counters_extract(mespack_rejects_extract_df, site_name=site_name, local_tz=site_tz)
                ## add machine speed
                mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df, 
                                            dim_df=machine_status_df.rename(columns={'Value':'Machine_Speed'}), 
                                            dim_id='Machine_Speed', fact_dttm='DateTime', dim_dttm='DateTime', 
                                            group_by=['site', 'line'])

                ## add agile flag if enabled
                if site_agile_enabled == True:
                    print(f'Agile is enabled for the site.')
                    print(f'Extracting data between: {start_time} and {end_time}')
                    agile_extract_df = historian.get_tag_values(site_servers['historian']['proficy']['server_name'], 
                                                                        start_time=start_time - timedelta(days=1),
                                                                        end_time=end_time,
                                                                        filter_name=site_agile_tags)
                    logging.info(f'Shape of agile flag extract: {agile_extract_df.shape}')

                    if agile_extract_df.shape[0] > 0:
                        agile_status_df = put_to_wonderware_format(agile_extract_df, inplace=False)
                        agile_status_df = agile_status_df.assign(DateTime = agile_status_df['DateTime'].dt.tz_convert(site_tz).dt.tz_localize(None),
                                                                Value = agile_status_df['Value'].astype(int))
                        agile_status_df = add_site_line_tag(agile_status_df, site=site_name)
                        agile_status_df = agile_status_df.drop(columns=['tag'])

                        agile_lines_mask = lines_dim_df['line_historian'].isin(site_lines_agile) & ~lines_dim_df['is_vec']
                        agile_legs_df = lines_dim_df.loc[agile_lines_mask, ['line_historian', 'leg']]
                        agile_status_df = agile_status_df.merge(agile_legs_df, left_on='line', right_on='line_historian', how='inner')
                        agile_status_df = (agile_status_df
                                .assign(line = agile_status_df['line'].str.cat(agile_status_df['leg']))
                                .drop(columns=['leg', 'line_historian']))

                        ## add agile flag
                        mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df, 
                                                    dim_df=agile_status_df.rename(columns={'Value':'Agile_Flag'}), 
                                                    dim_id='Agile_Flag', fact_dttm='DateTime', dim_dttm='DateTime', 
                                                    group_by=['site', 'line'])
                        mespack_rejects_df = mespack_rejects_df.assign(Agile_Flag = mespack_rejects_df['Agile_Flag'].fillna(0))
                        is_agile = True
                    else:
                        mespack_rejects_df = mespack_rejects_df.assign(Agile_Flag = 0)
                        is_agile = False
                else:
                    mespack_rejects_df = mespack_rejects_df.assign(Agile_Flag = 0)
                    is_agile = False

                ## add project flag if enabled
                if site_project_enabled == True:
                    print(f'Project is enabled for the site.')
                    print(f'Extracting data between: {start_time} and {end_time}')
                    project_extract_df = historian.get_tag_values(site_servers['historian']['proficy']['server_name'], 
                                                                  start_time=start_time - timedelta(days=1),
                                                                  end_time=end_time,
                                                                  filter_name=site_project_tags)
                    print(f'Shape of agile flag extract: {project_extract_df.shape}')

                    if project_extract_df.shape[0] > 0:
                        project_status_df = put_to_wonderware_format(project_extract_df, inplace=False)
                        project_status_df = project_status_df.assign(DateTime = project_status_df['DateTime'].dt.tz_convert(site_tz).dt.tz_localize(None),
                                                                    Value = project_status_df['Value'].astype(int))
                        project_status_df = add_site_line_tag(project_status_df, site=site_name)
                        project_status_df = project_status_df.drop(columns=['tag'])

                        ## add project tag
                        mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df, 
                                                    dim_df=project_status_df.rename(columns={'Value':'Project_Flag'}), 
                                                    dim_id='Project_Flag', fact_dttm='DateTime', dim_dttm='DateTime', 
                                                    group_by=['site', 'line'])
                        mespack_rejects_df = mespack_rejects_df.assign(Project_Flag = mespack_rejects_df['Project_Flag'].fillna(0))
                        is_project = True
                    else:
                        mespack_rejects_df = mespack_rejects_df.assign(Project_Flag = 0)
                        is_project = False
                else:
                    mespack_rejects_df = mespack_rejects_df.assign(Project_Flag = 0)
                    is_project = False

                ## add line state
                mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df.assign(line = mespack_rejects_df['line'].map(line_maping_dict)), 
                                            dim_df=line_state_map_df, 
                                            dim_id='line_state_id', fact_dttm='DateTime', dim_dttm='DateTime', 
                                            group_by=['site', 'line'])

                ## aggregating
                mespack_rejects_df = mespack_rejects_df.assign(DateTimeMin = mespack_rejects_df['DateTime'].dt.floor(pack_rejects_agg_freq))

                groupby_cols = ['site', 'line', 'DateTimeMin', 'tag', 
                                'Machine_Speed', 'Agile_Flag', 'Project_Flag', 'line_state_id']
                agg_dict = {'rejects_qty':'sum', 'DateTime':'min'}

                mespack_rejects_agg_df = mespack_rejects_df.groupby(groupby_cols).agg(agg_dict).reset_index()

                ## adding line_id and site_id
                rename_dict = {'DateTimeMin':'datetime', 'tag':'reject_type', 
                               'Machine_Speed':'machine_speed', 'DateTime':'start_time', 
                               'Agile_Flag':'agile_flag', 'Project_Flag':'project_flag'}
                mespack_rejects_agg_df = (mespack_rejects_agg_df
                                    .merge(lines_dim_df[['line_id', 'line']], on = 'line', how='inner')
                                    .merge(sites_dim_df[['site_id', 'site']], on = 'site', how='inner')
                                    .drop(columns=['site', 'line'])
                                    .rename(columns=rename_dict)
                                )

TRANSFORM: Shape of original paking rejects data: (441713, 3)
TRANSFORM: Step 1 - Removing cases where the counter values were not logged (NaNs) or negative
TRANSFORM: # removed records 0
TRANSFORM: Shape after step: (441713, 3)
TRANSFORM: Step 2 - Removing cases, where counter has dropped down in between two resets (historian outage issue)
TRANSFORM: # removed records 16
TRANSFORM: Shape after step: (441697, 4)
TRANSFORM: Step 3 - Removing cases where the counter was not changing
TRANSFORM: # removed records 4367
TRANSFORM: Shape after step: (437330, 5)
TRANSFORM: Step 4 - Removing cases where the counter reports zero
TRANSFORM: # removed records 14896
TRANSFORM: Shape after step: (422434, 5)
TRANSFORM: Shape after addition of columns: (422434, 6)


In [84]:
print('Transformed bag rejects data:')
print(mespack_rejects_agg_df.shape)
mespack_rejects_agg_df.head()

Transformed bag rejects data:
(25618, 10)


,datetime,reject_type,machine_speed,agile_flag,project_flag,line_state_id,rejects_qty,start_time,line_id,site_id
0,2023-02-20 16:10:00,Changeover_Reject_Total,40,0.0,0.0,23632/0,646.0,2023-02-20 16:18:38.259,54,3
1,2023-02-20 16:10:00,DS_Bottom_Suction,40,0.0,0.0,23632/0,6.0,2023-02-20 16:18:38.259,54,3
2,2023-02-20 16:10:00,DS_Filling_Disabled,40,0.0,0.0,23632/0,304.0,2023-02-20 16:18:38.259,54,3
3,2023-02-20 16:10:00,DS_Scale_Discharge,40,0.0,0.0,23632/0,3.0,2023-02-20 16:18:38.259,54,3
4,2023-02-20 16:10:00,US_Bottom_Suction,40,0.0,0.0,23632/0,5.0,2023-02-20 16:18:38.259,54,3


In [85]:
mespack_rejects_agg_df['start_time'].min(), mespack_rejects_agg_df['start_time'].max()

(Timestamp('2023-02-20 15:00:01.622000'),
 Timestamp('2023-03-17 00:10:00.294000'))

In [86]:
mespack_rejects_agg_df

,datetime,reject_type,machine_speed,agile_flag,project_flag,line_state_id,rejects_qty,start_time,line_id,site_id
0,2023-02-20 16:10:00,Changeover_Reject_Total,40,0.0,0.0,23632/0,646.0,2023-02-20 16:18:38.259,54,3
1,2023-02-20 16:10:00,DS_Bottom_Suction,40,0.0,0.0,23632/0,6.0,2023-02-20 16:18:38.259,54,3
2,2023-02-20 16:10:00,DS_Filling_Disabled,40,0.0,0.0,23632/0,304.0,2023-02-20 16:18:38.259,54,3
3,2023-02-20 16:10:00,DS_Scale_Discharge,40,0.0,0.0,23632/0,3.0,2023-02-20 16:18:38.259,54,3
4,2023-02-20 16:10:00,US_Bottom_Suction,40,0.0,0.0,23632/0,5.0,2023-02-20 16:18:38.259,54,3
...,...,...,...,...,...,...,...,...,...,...
25613,2023-02-21 11:50:00,DS_Filling_Disabled,35,0.0,0.0,23670/0,149.0,2023-02-21 11:56:18.004,62,3
25614,2023-02-21 11:50:00,US_Filling_Disabled,35,0.0,0.0,23670/0,149.0,2023-02-21 11:56:18.004,62,3
25615,2023-02-21 10:30:00,Changeover_Reject_Total,29,0.0,0.0,23671/0,244.0,2023-02-21 10:32:19.780,61,3
25616,2023-02-21 10:30:00,DS_Filling_Disabled,29,0.0,0.0,23671/0,122.0,2023-02-21 10:32:19.780,61,3


In [87]:
mespack_rejects_agg_df.line_id.unique()

array([54, 55, 58, 62, 61], dtype=int64)

In [88]:
mespack_rejects_agg_df[['datetime','line_state_id','line_id','site_id','reject_type','machine_speed']].duplicated().any()

False

In [89]:
mespack_rejects_agg_df[['datetime','line_state_id','line_id','site_id','reject_type','machine_speed']].isnull().any()

datetime         False
line_state_id    False
line_id          False
site_id          False
reject_type      False
machine_speed    False
dtype: bool

In [90]:
mespack_rejects_agg_df.reject_type.unique()

array(['Changeover_Reject_Total', 'DS_Bottom_Suction',
       'DS_Filling_Disabled', 'DS_Scale_Discharge', 'US_Bottom_Suction',
       'US_Filling_Disabled', 'US_Scale_Discharge', 'Film_Splice',
       '2D_Markem_Reject', 'Seal_Reject', 'Machine_Index',
       'DS_Top_Suction', 'US_Top_Suction', 'Missing_Gusset',
       'DS_Bag_Open', 'US_Bag_Open'], dtype=object)

In [91]:
mespack_rejects_agg_df

,datetime,reject_type,machine_speed,agile_flag,project_flag,line_state_id,rejects_qty,start_time,line_id,site_id
0,2023-02-20 16:10:00,Changeover_Reject_Total,40,0.0,0.0,23632/0,646.0,2023-02-20 16:18:38.259,54,3
1,2023-02-20 16:10:00,DS_Bottom_Suction,40,0.0,0.0,23632/0,6.0,2023-02-20 16:18:38.259,54,3
2,2023-02-20 16:10:00,DS_Filling_Disabled,40,0.0,0.0,23632/0,304.0,2023-02-20 16:18:38.259,54,3
3,2023-02-20 16:10:00,DS_Scale_Discharge,40,0.0,0.0,23632/0,3.0,2023-02-20 16:18:38.259,54,3
4,2023-02-20 16:10:00,US_Bottom_Suction,40,0.0,0.0,23632/0,5.0,2023-02-20 16:18:38.259,54,3
...,...,...,...,...,...,...,...,...,...,...
25613,2023-02-21 11:50:00,DS_Filling_Disabled,35,0.0,0.0,23670/0,149.0,2023-02-21 11:56:18.004,62,3
25614,2023-02-21 11:50:00,US_Filling_Disabled,35,0.0,0.0,23670/0,149.0,2023-02-21 11:56:18.004,62,3
25615,2023-02-21 10:30:00,Changeover_Reject_Total,29,0.0,0.0,23671/0,244.0,2023-02-21 10:32:19.780,61,3
25616,2023-02-21 10:30:00,DS_Filling_Disabled,29,0.0,0.0,23671/0,122.0,2023-02-21 10:32:19.780,61,3


In [92]:
mespack_rejects_agg_df.site_id.unique()

array([3], dtype=int64)

#### Load

For loading the data into datalabDB, we will use Upsert or update query. Based on provided primary key, Upsert query upserts dataframe to the target table in DB and insert query will be used to insert the new data which is only available in the new extract.
Following steps describe the process:
1. We will first take transformed bag rejects dataframe 'mespack_rejects_agg_df'. 
2. setup a database connection.
3. We will define a target table to upsert to in DB. Here, the target table is SUD_BAGGER_REJECTS.
3. Defining primary key for the upsert.
4. Defining template of query creating the temporary table in DB to be used for the upsert. We creats temp table in DB to insert transformed bag rejects dataframe. We are then upserting into the target table from the temp table.
                            

In [32]:
if site_enabled == True:
        if site_historian_source == 'proficy':
            # loading
            truncate_date = start_time + timedelta(days=site_history_buffer_days)
#             truncate_date =  start_time + timedelta(days=1)
            site_sud_id = sites_dim_df.loc[sites_dim_df['site'] == site_name, 'site_id'].iloc[0]

            mespack_rejects_agg_df = mespack_rejects_agg_df.loc[mespack_rejects_agg_df['datetime'] >= truncate_date]
            try:
                truncate_query = queries.TRUNCATE_REJECTS_TABLE.format(pack_rejects_table_nm, site_sud_id, truncate_date)
                execute_db_query(conn_string=datalab_db_conn, query=truncate_query, access_token=datalab_db_access_token)
                insert_df_to_mssql_db(df=mespack_rejects_agg_df,
                                    db_conn=datalab_db_conn,
                                    db_table=pack_rejects_table_nm,
                                    access_token=datalab_db_access_token)

                # updating etl status table 
                extract_overview_df = (mespack_rejects_agg_df.groupby(['site_id', 'line_id'])
                        .agg({'datetime':'max', 'line_state_id':'count'})
                        .reset_index()
                        .rename(columns={'line_state_id':'n_records', 'datetime':'start_time'}))
                new_last_available_dttm = extract_overview_df['start_time'].min()

                job_summary_record = get_run_summary_entry(job_id, job_nm, pack_rejects_table_nm,
                                                            job_step_start_dttm, datetime.now(),
                                                            extract_overview_df['n_records'].sum(),
                                                            np.nan, 'start_time', 
                                                            new_last_available_dttm, np.nan)
                insert_df_to_mssql_db(df=job_summary_record,
                          db_conn=datalab_db_conn,
                          db_table=job_summary_table_nm,
                          access_token=datalab_db_access_token)

            except Exception as e:
                extract_overview_df = pd.DataFrame()

                job_summary_record = get_run_summary_entry(job_id, job_nm, pack_production_table_nm,
                                                            job_step_start_dttm, datetime.now(),
                                                            0, np.nan, 'start_time', 
                                                            np.nan, np.nan, status='failed',
                                                            error_message=str(e))

                insert_df_to_mssql_db(df=job_summary_record,
                          db_conn=datalab_db_conn,
                          db_table=job_summary_table_nm,
                          access_token=datalab_db_access_token)    
        elif site_historian_source == 'wonderware':
            raise ValueError('Not implemented')

        else:
            err_msg = 'Incorrect historian system name. Should be one of ("proficy", "wonderware"). Adjust config file.'
            raise ValueError(err_msg)

else:
        pass
    


In [33]:
# if site_enabled == True:
#         if site_historian_source == 'proficy':
#             # loading
#             mespack_rejects_agg_df = mespack_rejects_agg_df.loc[mespack_rejects_agg_df['datetime'] >= start_time + timedelta(days=site_history_buffer_days)]
#             # mespack_rejects_agg_df = mespack_rejects_agg_df.loc[mespack_rejects_agg_df['datetime'] >= start_time + timedelta(days=1)]
#             try:
#                 upsert_data_to_datalab(data_frame=mespack_rejects_agg_df, 
#                             db_conn=datalab_db_conn, 
#                             db_access_token=datalab_db_access_token,
#                             target_table_nm=pack_rejects_table_nm, 
#                             primary_key=['line_state_id', 'site_id', 'line_id', 'datetime', 
#                                          'reject_type', 'machine_speed', 'agile_flag', 'project_flag'], 
#                             query_create_template=queries.CREATE_BAG_REJECTS_TMP)

#                 # updating etl status table 
#                 extract_overview_df = (mespack_rejects_agg_df.groupby(['site_id', 'line_id'])
#                         .agg({'datetime':'max', 'line_state_id':'count'})
#                         .reset_index()
#                         .rename(columns={'line_state_id':'n_records', 'datetime':'start_time'}))
#                 new_last_available_dttm = extract_overview_df['start_time'].min()

#                 job_summary_record = get_run_summary_entry(job_id, job_nm, pack_rejects_table_nm,
#                                                             job_step_start_dttm, datetime.now(),
#                                                             extract_overview_df['n_records'].sum(),
#                                                             np.nan, 'start_time', 
#                                                             new_last_available_dttm, np.nan)
#                 insert_df_to_mssql_db(df=job_summary_record,
#                           db_conn=datalab_db_conn,
#                           db_table=job_summary_table_nm,
#                           access_token=datalab_db_access_token)

#             except Exception as e:
#                 extract_overview_df = pd.DataFrame()

#                 job_summary_record = get_run_summary_entry(job_id, job_nm, pack_production_table_nm,
#                                                             job_step_start_dttm, datetime.now(),
#                                                             0, np.nan, 'start_time', 
#                                                             np.nan, np.nan, status='failed',
#                                                             error_message=str(e))

#                 insert_df_to_mssql_db(df=job_summary_record,
#                           db_conn=datalab_db_conn,
#                           db_table=job_summary_table_nm,
#                           access_token=datalab_db_access_token)    
#         elif site_historian_source == 'wonderware':
#             raise ValueError('Not implemented')

#         else:
#             err_msg = 'Incorrect historian system name. Should be one of ("proficy", "wonderware"). Adjust config file.'
#             raise ValueError(err_msg)

# else:
#         pass
    


### Extract, transform and load for generating SUD_BAGGER_TOTALS table.

Following are the ETL steps for generating SUD_BAGGER_TOTALS table. This steps are similar to the ETL steps used for creating SUD_BAGGER_REJECTS table.

#### Extract

In [93]:
job_step_start_dttm = datetime.now()

In [94]:
if site_enabled == True:
        if site_historian_source == 'proficy':
            # define extract window
            last_run_df = get_last_run_df(table_nm=pack_production_table_nm,
                        db_conn=get_db_connection(datalab_db_conn, datalab_db_access_token),
                        job_nm=job_nm)

            if not last_run_df.empty:
                last_available_dttm = last_run_df['last_dttm_available'][0]
                last_available_id = last_run_df['last_id_available'][0]
            else:
                last_available_dttm = None
                last_available_id = None

            start_dttm = datetime.now() if last_available_dttm is None else last_available_dttm
            start_dttm = start_dttm - timedelta(days=site_days_to_retake)
            start_time = start_dttm
            end_time = datetime.now()
            print(f'Extracting data between: {start_time} and {end_time}')

            

Extracting data between: 2023-03-06 15:23:46.360701 and 2023-03-16 15:23:46.360701


In [95]:
#start_time = start_time - timedelta(days=9, hours=12, minutes=0)
start_time = pd.to_datetime('2023-02-20 06:00:00')
end_time = datetime.now() + timedelta(days=1)
print(start_time, end_time)

2023-02-20 06:00:00 2023-03-17 15:23:48.033733


In [96]:
if site_enabled == True:
        if site_historian_source == 'proficy':
            ## extract total production/rejects
            mespack_total_rejects_extract_df = historian.get_tag_values(
                site_servers['historian']['proficy']['server_name'],
                start_time=start_time,
                end_time=end_time,
                filter_name=tags_total_prod + tags_total_rejects)
            print(f'Shape of bag production and total rejects extract: {mespack_total_rejects_extract_df.shape}')
            

Shape of bag production and total rejects extract: (1161691, 2)


In [97]:
print('Historian extract for bag production data:')
mespack_total_rejects_extract_df.head()

Historian extract for bag production data:


Value  \
Tag                                      Timestamp                                 
TAKF-PACKING.L113_Mespack_Good_Bag_Total 2023-03-16 10:58:25.828000+00:00      0   
                                         2023-03-16 10:35:48.775000+00:00      0   
                                         2023-03-16 09:50:18.001000+00:00      0   
                                         2023-03-16 09:37:05.004000+00:00  17676   
                                         2023-03-16 09:37:01.988000+00:00  17675   

                                                                           Quality  
Tag                                      Timestamp                                  
TAKF-PACKING.L113_Mespack_Good_Bag_Total 2023-03-16 10:58:25.828000+00:00        3  
                                         2023-03-16 10:35:48.775000+00:00        3  
                                         2023-03-16 09:50:18.001000+00:00        3  
                                         2023-03-16 09:37:05.004000+00:00        3  
                                         2023-03-16 09:37:01.988000+00:00        3

### Transform

In [98]:
if site_enabled == True:
        if site_historian_source == 'proficy':
            # transform
                mespack_total_rejects_df = transform_counters_extract(mespack_total_rejects_extract_df, 
                                                    site_name=site_name, counter_name='qty', local_tz=site_tz)
                ## add machine speed
                mespack_total_rejects_df = add_dim_key_time(fact_df=mespack_total_rejects_df, 
                                            dim_df=machine_status_df.rename(columns={'Value':'Machine_Speed'}), 
                                            dim_id='Machine_Speed', fact_dttm='DateTime', dim_dttm='DateTime', 
                                            group_by=['site', 'line'])

                ## add agile flag if enabled
                if is_agile:
                    mespack_total_rejects_df = add_dim_key_time(fact_df=mespack_total_rejects_df, 
                                                dim_df=agile_status_df.rename(columns={'Value':'Agile_Flag'}), 
                                                dim_id='Agile_Flag', fact_dttm='DateTime', dim_dttm='DateTime', 
                                                group_by=['site', 'line'])
                    mespack_total_rejects_df = mespack_total_rejects_df.assign(Agile_Flag = mespack_total_rejects_df['Agile_Flag'].fillna(0))
                else:
                    mespack_total_rejects_df = mespack_total_rejects_df.assign(Agile_Flag = 0)

                ## add projects flag if enabled
                if is_project:
                    mespack_total_rejects_df = add_dim_key_time(fact_df=mespack_total_rejects_df, 
                                                dim_df=project_status_df.rename(columns={'Value':'Project_Flag'}), 
                                                dim_id='Project_Flag', fact_dttm='DateTime', dim_dttm='DateTime', 
                                                group_by=['site', 'line'])
                    mespack_total_rejects_df = mespack_total_rejects_df.assign(Project_Flag = mespack_total_rejects_df['Project_Flag'].fillna(0))
                else:
                    mespack_total_rejects_df = mespack_total_rejects_df.assign(Project_Flag = 0)

                ## adding line state
                mespack_total_rejects_df = add_dim_key_time(
                    fact_df=mespack_total_rejects_df.assign(line = mespack_total_rejects_df['line'].map(line_maping_dict)), 
                    dim_df=line_state_map_df, 
                    dim_id='line_state_id', fact_dttm='DateTime', dim_dttm='DateTime', 
                    group_by=['site', 'line'])

                ## aggregating
                mespack_total_rejects_df = mespack_total_rejects_df.assign(DateTimeHour = mespack_total_rejects_df['DateTime'].dt.floor(pack_production_agg_freq))

                groupby_cols = ['site', 'line', 'DateTimeHour', 'tag', 
                                'Machine_Speed', 'Agile_Flag', 'Project_Flag', 'line_state_id']
                agg_dict = {'qty':'sum', 'DateTime':'min'}

                mespack_total_rejects_agg_df = mespack_total_rejects_df.groupby(groupby_cols).agg(agg_dict).reset_index()

                ## transposing
                mespack_total_rejects_agg_df = transpose_data_frame(data_frame=mespack_total_rejects_agg_df,
                                                rows=['site', 'line', 'DateTimeHour', 'Machine_Speed', 'Agile_Flag', 
                                                      'Project_Flag', 'line_state_id', 'DateTime'],
                                                transpose_on='tag')
                if site_cd in ['ami', 'url']:
                    mespack_total_rejects_agg_df = mespack_total_rejects_agg_df.assign(Changeover_Reject_Total = 0)

                agg_dict = {'Changeover_Reject_Total':'sum', 'Good_Bag_Total':'sum', 
                            'Reject_Bag_Total':'sum', 'DateTime':'min'}
                mespack_total_rejects_agg_df = (mespack_total_rejects_agg_df
                                                .groupby(['site', 'line', 'DateTimeHour', 
                                                          'Machine_Speed', 'Agile_Flag', 'Project_Flag',
                                                          'line_state_id'])
                                                .agg(agg_dict))
                mespack_total_rejects_agg_df = mespack_total_rejects_agg_df.reset_index()

                ## adding line_id and site_id
                rename_dict = {'DateTimeHour':'datetime', 'Machine_Speed':'machine_speed', 
                               'Agile_Flag':'agile_flag', 'Project_Flag':'project_flag', 'DateTime':'start_time'}
                mespack_total_rejects_agg_df = (mespack_total_rejects_agg_df
                                    .merge(lines_dim_df[['line_id', 'line']], on = 'line', how='inner')
                                    .merge(sites_dim_df[['site_id', 'site']], on = 'site', how='inner')
                                    .drop(columns=['site', 'line'])
                                    .rename(columns=rename_dict)
                                )


TRANSFORM: Shape of original paking rejects data: (1161667, 3)
TRANSFORM: Step 1 - Removing cases where the counter values were not logged (NaNs) or negative
TRANSFORM: # removed records 0
TRANSFORM: Shape after step: (1161667, 3)
TRANSFORM: Step 2 - Removing cases, where counter has dropped down in between two resets (historian outage issue)
TRANSFORM: # removed records 5
TRANSFORM: Shape after step: (1161662, 4)
TRANSFORM: Step 3 - Removing cases where the counter was not changing
TRANSFORM: # removed records 1682
TRANSFORM: Shape after step: (1159980, 5)
TRANSFORM: Step 4 - Removing cases where the counter reports zero
TRANSFORM: # removed records 929
TRANSFORM: Shape after step: (1159051, 5)
TRANSFORM: Shape after addition of columns: (1159051, 6)


In [99]:
print('Transformed bag production data:')
print(mespack_total_rejects_agg_df.shape)
mespack_total_rejects_agg_df.head()

Transformed bag production data:
(8789, 11)


,datetime,machine_speed,agile_flag,project_flag,line_state_id,Changeover_Reject_Total,Good_Bag_Total,Reject_Bag_Total,start_time,line_id,site_id
0,2023-02-20 16:00:00,40,0.0,0.0,23632/0,646.0,0.0,646.0,2023-02-20 16:18:38.259,54,3
1,2023-02-20 21:00:00,10,0.0,0.0,23632/0,2.0,0.0,0.0,2023-02-20 21:25:32.454,54,3
2,2023-02-20 21:00:00,40,0.0,0.0,23632/0,16.0,0.0,18.0,2023-02-20 21:25:37.425,54,3
3,2023-02-20 21:30:00,40,0.0,0.0,23632/0,154.0,0.0,154.0,2023-02-20 21:40:36.515,54,3
4,2023-02-20 22:30:00,40,0.0,0.0,23632/0,2.0,0.0,2.0,2023-02-20 22:34:30.511,54,3


In [100]:
mespack_total_rejects_agg_df[['datetime','line_state_id','line_id','site_id','machine_speed']].duplicated().any()

False

In [101]:
mespack_total_rejects_agg_df[['datetime','line_state_id','line_id','site_id','machine_speed']].isnull().any()

datetime         False
line_state_id    False
line_id          False
site_id          False
machine_speed    False
dtype: bool

In [102]:
mespack_total_rejects_agg_df.line_id.unique()

array([54, 55, 58, 62, 61], dtype=int64)

In [103]:
mespack_total_rejects_agg_df.site_id.unique()

array([3], dtype=int64)

### Load

In [17]:
truncate_date = start_time + timedelta(days=site_history_buffer_days)
print(truncate_date)

2023-03-09 19:00:00


In [18]:
start_time

Timestamp('2023-03-06 19:00:00')

In [42]:
if site_enabled == True:
        if site_historian_source == 'proficy':
            #loading
#             truncate_date =  start_time + timedelta(days=1)
            truncate_date = start_time + timedelta(days=site_history_buffer_days)
            site_sud_id = sites_dim_df.loc[sites_dim_df['site'] == site_name, 'site_id'].iloc[0]
            mespack_total_rejects_agg_df = mespack_total_rejects_agg_df.loc[mespack_total_rejects_agg_df['datetime'] >= truncate_date]
            
            try:
                truncate_query = queries.TRUNCATE_REJECTS_TABLE.format(pack_production_table_nm, site_sud_id, truncate_date)
                execute_db_query(conn_string=datalab_db_conn, query=truncate_query, access_token=datalab_db_access_token)
                insert_df_to_mssql_db(df=mespack_total_rejects_agg_df,
                                    db_conn=datalab_db_conn,
                                    db_table=pack_production_table_nm,
                                    access_token=datalab_db_access_token)

                # updating etl status table 
                extract_overview_df = (mespack_total_rejects_agg_df.groupby(['site_id', 'line_id'])
                                        .agg({'datetime':'max', 'line_state_id':'count'})
                                        .reset_index()
                                        .rename(columns={'line_state_id':'n_records', 'datetime':'start_time'}))
                new_last_available_dttm = extract_overview_df['start_time'].min()

                job_summary_record = get_run_summary_entry(job_id, job_nm, pack_production_table_nm,
                                                            job_step_start_dttm, datetime.now(),
                                                            extract_overview_df['n_records'].sum(),
                                                            np.nan, 'start_time', 
                                                            new_last_available_dttm, np.nan)
                insert_df_to_mssql_db(df=job_summary_record,
                          db_conn=datalab_db_conn,
                          db_table=job_summary_table_nm,
                          access_token=datalab_db_access_token)
            except Exception as e:
                extract_overview_df = pd.DataFrame()

                job_summary_record = get_run_summary_entry(job_id, job_nm, pack_production_table_nm,
                                                            job_step_start_dttm, datetime.now(),
                                                            0, np.nan, 'start_time', 
                                                            np.nan, np.nan, status='failed',
                                                            error_message=str(e))

                insert_df_to_mssql_db(df=job_summary_record,
                          db_conn=datalab_db_conn,
                          db_table=job_summary_table_nm,
                          access_token=datalab_db_access_token)    
        elif site_historian_source == 'wonderware':
            raise ValueError('Not implemented')

        else:
            err_msg = 'Incorrect historian system name. Should be one of ("proficy", "wonderware"). Adjust config file.'
            raise ValueError(err_msg)

else:
        pass
    

### Complete ETL process

The execution of complete extract, transform, and load process for both tables SUB_BAGGER_TOTALS and SUD_BAGGER_REJECTS is as follows:

In [43]:
# if site_enabled == True:
#         if site_historian_source == 'proficy':
#             # set up

#             non_agile_tags = [t for t in site_tags if t not in site_historian_tags['tags_agile'] + site_historian_tags['tags_projects']]
#             agile_tags = [t for t in site_tags if t in site_historian_tags['tags_agile']]
#             project_tags = [t for t in site_tags if t in site_historian_tags['tags_projects']]

#             site_line_tags = get_tags_list(lines=site_lines, sensors=non_agile_tags, sep='_', 
#                                            topic=site_servers['historian']['proficy']['topic'])
#             if agile_tags:
#                 site_agile_tags = get_tags_list(lines=site_lines_agile, sensors=agile_tags, sep='_', 
#                                                 topic=site_servers['historian']['proficy']['topic'])
#             else: 
#                 site_agile_tags = []
            
#             if project_tags:
#                 site_project_tags = get_tags_list(lines=site_lines, sensors=project_tags, sep='_', 
#                                                  topic=site_servers['historian']['proficy']['topic'])
#             else: 
#                 site_project_tags = []

#             index_tags = [t for t in site_line_tags if '_'.join(t.split('_')[1:]) in site_historian_tags['tags_index']]
#             tags_total_prod = [t for t in site_line_tags if t.endswith('Good_Bag_Total')]
#             tags_total_rejects = [t for t in site_line_tags if not t.endswith('Good_Bag_Total') and 'total' in t.lower()]
#             tags_rejects = [t for t in site_line_tags if (t not in index_tags + tags_total_prod + tags_total_rejects) 
#                                                         or 'changeover' in t.lower()] 

#             historian.use_context('REST', 
#                     client_id='historian_public_rest_api', 
#                     client_password='publicapisecret',
#                     app_id='sudanalytics.im', 
#                     app_password='phoenix2021SUD', # change to environmental variable
#                     port=site_servers['historian']['proficy']['port'],
#                     verify_certificate=True)
            
#             # define extract window
#             last_run_df = get_last_run_df(table_nm=pack_rejects_table_nm,
#                         db_conn=get_db_connection(datalab_db_conn, datalab_db_access_token),
#                         job_nm=job_nm)

#             if not last_run_df.empty:
#                 last_available_dttm = last_run_df['last_dttm_available'][0]
#                 last_available_id = last_run_df['last_id_available'][0]
#             else:
#                 last_available_dttm = None
#                 last_available_id = None

#             start_dttm = datetime.now() if last_available_dttm is None else last_available_dttm
#             start_dttm = start_dttm - timedelta(days=site_history_depth)
#             start_time = start_dttm
#             end_time = datetime.now()
#             print(f'Extracting data between: {start_time} and {end_time}')

#             try:
#                 # extract
#                 ## extracting machine status tags
#                 machine_status_extract_df = historian.get_tag_values(
#                     site_servers['historian']['proficy']['server_name'],
#                     start_time=start_time - timedelta(days=1),
#                     end_time=end_time,
#                     filter_name=index_tags)
#                 print(f'Shape of machine speed extract: {machine_status_extract_df.shape}')
                
#                 machine_status_df = put_to_wonderware_format(machine_status_extract_df, inplace=False)
#                 machine_status_df = machine_status_df.assign(DateTime = machine_status_df['DateTime'].dt.tz_convert(site_tz).dt.tz_localize(None))
#                 machine_status_df = add_site_line_tag(machine_status_df, site=site_name)
#                 machine_status_df = machine_status_df.drop(columns=['tag'])

#                 ## extract rejects
#                 mespack_rejects_extract_df = historian.get_tag_values(
#                     site_servers['historian']['proficy']['server_name'],
#                     start_time=start_time,
#                     end_time=end_time,
#                     filter_name=tags_rejects)
#                 logging.info(f'Shape of bag rejects extract: {mespack_rejects_extract_df.shape}')

#                 ## extract line state
#                 extract_dttm = datetime.strftime(start_time.date(), '%Y%m%d %H:%M:%S') 
#                 query_to_execute = queries.EXTRACT_LINE_STATE.format(extract_dttm, tuple(site_lines_dim))

#                 line_state_map_df = pd.read_sql(sql=query_to_execute, con=get_db_connection(datalab_db_conn, datalab_db_access_token))
#                 print(f'Line state was extracted, shape: {line_state_map_df.shape}')

#                 index_cols = ['line_state_id', 'line', 'site']
#                 line_state_map_df = (line_state_map_df
#                                     .set_index(index_cols)
#                                     .stack()
#                                     .to_frame('DateTime')
#                                     .reset_index()
#                                     .drop(columns=[f'level_{len(index_cols)}'])
#                                     )
#                 # transform
#                 mespack_rejects_df = transform_counters_extract(mespack_rejects_extract_df, site_name=site_name, local_tz=site_tz)
#                 ## add machine speed
#                 mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df, 
#                                             dim_df=machine_status_df.rename(columns={'Value':'Machine_Speed'}), 
#                                             dim_id='Machine_Speed', fact_dttm='DateTime', dim_dttm='DateTime', 
#                                             group_by=['site', 'line'])

#                 ## add agile flag if enabled
#                 if site_agile_enabled == True:
#                     print(f'Agile is enabled for the site.')
#                     print(f'Extracting data between: {start_time} and {end_time}')
#                     agile_extract_df = historian.get_tag_values(site_servers['historian']['proficy']['server_name'], 
#                                                                         start_time=start_time - timedelta(days=1),
#                                                                         end_time=end_time,
#                                                                         filter_name=site_agile_tags)
#                     print(f'Shape of agile flag extract: {agile_extract_df.shape}')

#                     if agile_extract_df.shape[0] > 0:
#                         agile_status_df = put_to_wonderware_format(agile_extract_df, inplace=False)
#                         agile_status_df = agile_status_df.assign(DateTime = agile_status_df['DateTime'].dt.tz_convert(site_tz).dt.tz_localize(None),
#                                                                 Value = agile_status_df['Value'].astype(int))
#                         agile_status_df = add_site_line_tag(agile_status_df, site=site_name)
#                         agile_status_df = agile_status_df.drop(columns=['tag'])

#                         agile_lines_mask = lines_dim_df['line_historian'].isin(site_lines_agile) & ~lines_dim_df['is_vec']
#                         agile_legs_df = lines_dim_df.loc[agile_lines_mask, ['line_historian', 'leg']]
#                         agile_status_df = agile_status_df.merge(agile_legs_df, left_on='line', right_on='line_historian', how='inner')
#                         agile_status_df = (agile_status_df
#                                 .assign(line = agile_status_df['line'].str.cat(agile_status_df['leg']))
#                                 .drop(columns=['leg', 'line_historian']))

#                         ## add agile flag
#                         mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df, 
#                                                     dim_df=agile_status_df.rename(columns={'Value':'Agile_Flag'}), 
#                                                     dim_id='Agile_Flag', fact_dttm='DateTime', dim_dttm='DateTime', 
#                                                     group_by=['site', 'line'])
#                         mespack_rejects_df = mespack_rejects_df.assign(Agile_Flag = mespack_rejects_df['Agile_Flag'].fillna(0))
#                         is_agile = True
#                     else:
#                         mespack_rejects_df = mespack_rejects_df.assign(Agile_Flag = 0)
#                         is_agile = False
#                 else:
#                     mespack_rejects_df = mespack_rejects_df.assign(Agile_Flag = 0)
#                     is_agile = False

#                 ## add project flag if enabled
#                 if site_project_enabled == True:
#                     print(f'Project is enabled for the site.')
#                     print(f'Extracting data between: {start_time} and {end_time}')
#                     project_extract_df = historian.get_tag_values(site_servers['historian']['proficy']['server_name'], 
#                                                                   start_time=start_time - timedelta(days=1),
#                                                                   end_time=end_time,
#                                                                   filter_name=site_project_tags)
#                     print(f'Shape of agile flag extract: {project_extract_df.shape}')

#                     if project_extract_df.shape[0] > 0:
#                         project_status_df = put_to_wonderware_format(project_extract_df, inplace=False)
#                         project_status_df = project_status_df.assign(DateTime = project_status_df['DateTime'].dt.tz_convert(site_tz).dt.tz_localize(None),
#                                                                     Value = project_status_df['Value'].astype(int))
#                         project_status_df = add_site_line_tag(project_status_df, site=site_name)
#                         project_status_df = project_status_df.drop(columns=['tag'])

#                         ## add project tag
#                         mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df, 
#                                                     dim_df=project_status_df.rename(columns={'Value':'Project_Flag'}), 
#                                                     dim_id='Project_Flag', fact_dttm='DateTime', dim_dttm='DateTime', 
#                                                     group_by=['site', 'line'])
#                         mespack_rejects_df = mespack_rejects_df.assign(Project_Flag = mespack_rejects_df['Project_Flag'].fillna(0))
#                         is_project = True
#                     else:
#                         mespack_rejects_df = mespack_rejects_df.assign(Project_Flag = 0)
#                         is_project = False
#                 else:
#                     mespack_rejects_df = mespack_rejects_df.assign(Project_Flag = 0)
#                     is_project = False

#                 ## add line state
#                 mespack_rejects_df = add_dim_key_time(fact_df=mespack_rejects_df.assign(line = mespack_rejects_df['line'].map(line_maping_dict)), 
#                                             dim_df=line_state_map_df, 
#                                             dim_id='line_state_id', fact_dttm='DateTime', dim_dttm='DateTime', 
#                                             group_by=['site', 'line'])

#                 ## aggregating
#                 mespack_rejects_df = mespack_rejects_df.assign(DateTimeMin = mespack_rejects_df['DateTime'].dt.floor(pack_rejects_agg_freq))

#                 groupby_cols = ['site', 'line', 'DateTimeMin', 'tag', 
#                                 'Machine_Speed', 'Agile_Flag', 'Project_Flag', 'line_state_id']
#                 agg_dict = {'rejects_qty':'sum', 'DateTime':'min'}

#                 mespack_rejects_agg_df = mespack_rejects_df.groupby(groupby_cols).agg(agg_dict).reset_index()

#                 ## adding line_id and site_id
#                 rename_dict = {'DateTimeMin':'datetime', 'tag':'reject_type', 
#                                'Machine_Speed':'machine_speed', 'DateTime':'start_time', 
#                                'Agile_Flag':'agile_flag', 'Project_Flag':'project_flag'}
#                 mespack_rejects_agg_df = (mespack_rejects_agg_df
#                                     .merge(lines_dim_df[['line_id', 'line']], on = 'line', how='inner')
#                                     .merge(sites_dim_df[['site_id', 'site']], on = 'site', how='inner')
#                                     .drop(columns=['site', 'line'])
#                                     .rename(columns=rename_dict)
#                                 )
#                 # loading
#                 upsert_data_to_datalab(data_frame=mespack_rejects_agg_df, 
#                             db_conn=datalab_db_conn, 
#                             db_access_token=datalab_db_access_token,
#                             target_table_nm=pack_rejects_table_nm, 
#                             primary_key=['line_state_id', 'site_id', 'line_id', 'datetime', 
#                                          'reject_type', 'machine_speed', 'agile_flag', 'project_flag'], 
#                             query_create_template=queries.CREATE_BAG_REJECTS_TMP)

#                 # updating etl status table 
#                 extract_overview_df = (mespack_rejects_agg_df.groupby(['site_id', 'line_id'])
#                                         .agg({'start_time':'max', 'line_state_id':'count'})
#                                         .reset_index()
#                                         .rename(columns={'line_state_id':'n_records'}))
#                 new_last_available_dttm = extract_overview_df['start_time'].min()

#                 job_summary_record = get_run_summary_entry(job_id, job_nm, pack_rejects_table_nm,
#                                                             job_step_start_dttm, datetime.now(),
#                                                             extract_overview_df['n_records'].sum(),
#                                                             np.nan, 'start_time', 
#                                                             new_last_available_dttm, np.nan)
#                 insert_df_to_mssql_db(df=job_summary_record,
#                           db_conn=datalab_db_conn,
#                           db_table=job_summary_table_nm,
#                           access_token=datalab_db_access_token)

#             except Exception as e:
#                 extract_overview_df = pd.DataFrame()
#                 agile_extract_df = pd.DataFrame()
#                 project_extract_df = pd.DataFrame()

#                 job_summary_record = get_run_summary_entry(job_id, job_nm, pack_rejects_table_nm,
#                                                             job_step_start_dttm, datetime.now(),
#                                                             0, np.nan, 'start_time', 
#                                                             np.nan, np.nan, status='failed',
#                                                             error_message=str(e))

#                 insert_df_to_mssql_db(df=job_summary_record,
#                           db_conn=datalab_db_conn,
#                           db_table=job_summary_table_nm,
#                           access_token=datalab_db_access_token)


#             # define extract window
#             last_run_df = get_last_run_df(table_nm=pack_production_table_nm,
#                         db_conn=get_db_connection(datalab_db_conn, datalab_db_access_token),
#                         job_nm=job_nm)

#             if not last_run_df.empty:
#                 last_available_dttm = last_run_df['last_dttm_available'][0]
#                 last_available_id = last_run_df['last_id_available'][0]
#             else:
#                 last_available_dttm = None
#                 last_available_id = None

#             start_dttm = datetime.now() if last_available_dttm is None else last_available_dttm
#             start_dttm = start_dttm - timedelta(days=site_history_depth)
#             start_time = start_dttm
#             end_time = datetime.now()
#             print(f'Extracting data between: {start_time} and {end_time}')

#             try:
#                 # extract totals
#                 ## extract total production/rejects
#                 mespack_total_rejects_extract_df = historian.get_tag_values(
#                     site_servers['historian']['proficy']['server_name'],
#                     start_time=start_time,
#                     end_time=end_time,
#                     filter_name=tags_total_prod + tags_total_rejects)
#                 print(f'Shape of bag production and total rejects extract: {mespack_total_rejects_extract_df.shape}')
                
#                 # transform
#                 mespack_total_rejects_df = transform_counters_extract(mespack_total_rejects_extract_df, 
#                                                     site_name=site_name, counter_name='qty', local_tz=site_tz)
#                 ## add machine speed
#                 mespack_total_rejects_df = add_dim_key_time(fact_df=mespack_total_rejects_df, 
#                                             dim_df=machine_status_df.rename(columns={'Value':'Machine_Speed'}), 
#                                             dim_id='Machine_Speed', fact_dttm='DateTime', dim_dttm='DateTime', 
#                                             group_by=['site', 'line'])

#                 ## add agile flag if enabled
#                 if is_agile:
#                     mespack_total_rejects_df = add_dim_key_time(fact_df=mespack_total_rejects_df, 
#                                                 dim_df=agile_status_df.rename(columns={'Value':'Agile_Flag'}), 
#                                                 dim_id='Agile_Flag', fact_dttm='DateTime', dim_dttm='DateTime', 
#                                                 group_by=['site', 'line'])
#                     mespack_total_rejects_df = mespack_total_rejects_df.assign(Agile_Flag = mespack_total_rejects_df['Agile_Flag'].fillna(0))
#                 else:
#                     mespack_total_rejects_df = mespack_total_rejects_df.assign(Agile_Flag = 0)

#                 ## add projects flag if enabled
#                 if is_project:
#                     mespack_total_rejects_df = add_dim_key_time(fact_df=mespack_total_rejects_df, 
#                                                 dim_df=project_status_df.rename(columns={'Value':'Project_Flag'}), 
#                                                 dim_id='Project_Flag', fact_dttm='DateTime', dim_dttm='DateTime', 
#                                                 group_by=['site', 'line'])
#                     mespack_total_rejects_df = mespack_total_rejects_df.assign(Project_Flag = mespack_total_rejects_df['Project_Flag'].fillna(0))
#                 else:
#                     mespack_total_rejects_df = mespack_total_rejects_df.assign(Project_Flag = 0)

#                 ## adding line state
#                 mespack_total_rejects_df = add_dim_key_time(
#                     fact_df=mespack_total_rejects_df.assign(line = mespack_total_rejects_df['line'].map(line_maping_dict)), 
#                     dim_df=line_state_map_df, 
#                     dim_id='line_state_id', fact_dttm='DateTime', dim_dttm='DateTime', 
#                     group_by=['site', 'line'])

#                 ## aggregating
#                 mespack_total_rejects_df = mespack_total_rejects_df.assign(DateTimeHour = mespack_total_rejects_df['DateTime'].dt.floor(pack_production_agg_freq))

#                 groupby_cols = ['site', 'line', 'DateTimeHour', 'tag', 
#                                 'Machine_Speed', 'Agile_Flag', 'Project_Flag', 'line_state_id']
#                 agg_dict = {'qty':'sum', 'DateTime':'min'}

#                 mespack_total_rejects_agg_df = mespack_total_rejects_df.groupby(groupby_cols).agg(agg_dict).reset_index()

#                 ## transposing
#                 mespack_total_rejects_agg_df = transpose_data_frame(data_frame=mespack_total_rejects_agg_df,
#                                                 rows=['site', 'line', 'DateTimeHour', 'Machine_Speed', 'Agile_Flag', 
#                                                       'Project_Flag', 'line_state_id', 'DateTime'],
#                                                 transpose_on='tag')

#                 agg_dict = {'Changeover_Reject_Total':'sum', 'Good_Bag_Total':'sum', 
#                             'Reject_Bag_Total':'sum', 'DateTime':'min'}
#                 mespack_total_rejects_agg_df = (mespack_total_rejects_agg_df
#                                                 .groupby(['site', 'line', 'DateTimeHour', 
#                                                           'Machine_Speed', 'Agile_Flag', 'Project_Flag',
#                                                           'line_state_id'])
#                                                 .agg(agg_dict))
#                 mespack_total_rejects_agg_df = mespack_total_rejects_agg_df.reset_index()

#                 ## adding line_id and site_id
#                 rename_dict = {'DateTimeHour':'datetime', 'Machine_Speed':'machine_speed', 
#                                'Agile_Flag':'agile_flag', 'Project_Flag':'project_flag', 'DateTime':'start_time'}
#                 mespack_total_rejects_agg_df = (mespack_total_rejects_agg_df
#                                     .merge(lines_dim_df[['line_id', 'line']], on = 'line', how='inner')
#                                     .merge(sites_dim_df[['site_id', 'site']], on = 'site', how='inner')
#                                     .drop(columns=['site', 'line'])
#                                     .rename(columns=rename_dict)
#                                 )

#                 upsert_data_to_datalab(data_frame=mespack_total_rejects_agg_df, 
#                             db_conn=datalab_db_conn, 
#                             db_access_token=datalab_db_access_token,
#                             target_table_nm=pack_production_table_nm, 
#                             primary_key=['line_state_id', 'site_id', 'line_id', 'datetime', 
#                                          'machine_speed', 'agile_flag', 'project_flag'], 
#                             query_create_template=queries.CREATE_BAG_TOTALS_TMP)

#                 # updating etl status table 
#                 extract_overview_df = (mespack_total_rejects_agg_df.groupby(['site_id', 'line_id'])
#                                         .agg({'start_time':'max', 'line_state_id':'count'})
#                                         .reset_index()
#                                         .rename(columns={'line_state_id':'n_records'}))
#                 new_last_available_dttm = extract_overview_df['start_time'].min()

#                 job_summary_record = get_run_summary_entry(job_id, job_nm, pack_production_table_nm,
#                                                             job_step_start_dttm, datetime.now(),
#                                                             extract_overview_df['n_records'].sum(),
#                                                             np.nan, 'start_time', 
#                                                             new_last_available_dttm, np.nan)
#                 insert_df_to_mssql_db(df=job_summary_record,
#                           db_conn=datalab_db_conn,
#                           db_table=job_summary_table_nm,
#                           access_token=datalab_db_access_token)

#             except Exception as e:
#                 extract_overview_df = pd.DataFrame()

#                 job_summary_record = get_run_summary_entry(job_id, job_nm, pack_production_table_nm,
#                                                             job_step_start_dttm, datetime.now(),
#                                                             0, np.nan, 'start_time', 
#                                                             np.nan, np.nan, status='failed',
#                                                             error_message=str(e))

#                 insert_df_to_mssql_db(df=job_summary_record,
#                           db_conn=datalab_db_conn,
#                           db_table=job_summary_table_nm,
#                           access_token=datalab_db_access_token)

#         elif site_historian_source == 'wonderware':
#             raise ValueError('Not implemented')

#         else:
#             err_msg = 'Incorrect historian system name. Should be one of ("proficy", "wonderware"). Adjust config file.'
#             raise ValueError(err_msg)

# else:
#         pass


# #refresh_pbi_dataset(api_url=pbi_refresh_url, app_name=pbi_refresh_app)




In [ ]:
refresh_pbi_dataset(api_url=pbi_refresh_url, app_name=pbi_refresh_app)